# Fit Model on full Dev Set

- Use model config derrived for each of self/parent cohorts in exploration to fit a single model
- Extract coefs from each trained self/parent model

## Set Up

In [ ]:
import sys, os

sys.path.append(os.path.abspath("../"))

from src.config import BASE_PATH, SEED
from src.utils import col_config_sensitivity
from src.coef_visual import plot_heatmap
from src.residual_func import (
    build_model,
    get_saliency_map,
)

In [ ]:
shared_dict = {
    "num_layers": 3,
    "max_iter": 50000,
    "seed": SEED,
    "linear_type": "ridge",
}
parent_config = {
    "impact_type": "parent",
    "quest_cols": col_config_sensitivity["parent_quest"],
    "agg_type": "max",
    "alpha2": 10.0,
    "alpha3": 5.0,
    "desc_seg_type": "pt_noshort",
    **shared_dict,
}
self_config = {
    "impact_type": "self",
    "quest_cols": col_config_sensitivity["self_quest"],
    "agg_type": "block",
    "alpha2": 40.0,
    "alpha3": 0.5,
    "desc_seg_type": "turns",
    **shared_dict,
}

## Build Model

In [ ]:
data_dir = BASE_PATH / "data"
llama_model = "Llama-3.3-70B-Instruct"
model_parent, coef_parent, df_parent = build_model(parent_config, llama_model, data_dir)
model_self, coef_self, df_self = build_model(self_config, llama_model, data_dir)

In [ ]:
coef_parent = coef_parent.rename(columns={"coef": "coef_parent"})
coef_self = coef_self.rename(columns={"coef": "coef_self"})
coef_combined = coef_parent.join(coef_self)
plot_heatmap(
    coef_combined,
    title="Learned Coef weights for final parent and self models",
    vmin=-2.5,
    vmax=2.5,
    xlabel="Cohort",
)

# Saliency Sanity Checks

In [ ]:
saliency_self = get_saliency_map(self_config, model_self, df_self, coef_self)

In [ ]:
saliency_parent = get_saliency_map(parent_config, model_parent, df_parent, coef_parent)

# LMG Feat Imp

- Randomly sample order of text outcomes n times
- Iteratively compute incremental r^2 after adding each outcome
- Avg across all 1000 samples to get avg contribution to accuracy

**TODO:**
- Improve parallelization

### Single Train-Test

In [ ]:
from src.Regressor import ResidualRegressor
from src.utils import expand_text
from src.lmg_imp import get_perm_feat_imp

# Get X/y train/test sets
target_col = f"impact_score_{parent_config['impact_type']}"
prelim_model = ResidualRegressor(**parent_config, two_layer_base=None)
df_parent_expand = expand_text(
    df=df_parent,
    agg_type=parent_config["agg_type"],
    desc_seg_type=parent_config["desc_seg_type"],
)
# X
X_train = df_parent_expand.drop(target_col, axis=1)
X_test = X_train
# y
y_train = df_parent_expand[target_col].to_numpy()
y_test = y_train

# Run feat imp
res_df = get_perm_feat_imp(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_config_dict=parent_config,
    n_samples=50,
    random_state=SEED,
)
res_df

### Via 10-repeated stratified CV

In [ ]:
from src.utils import get_df_path
from src.lmg_imp import lmg_imp_wrapper
from src.swarm import create_lmg_cmd, run_lmg_swarm

In [ ]:
cmd_list = []
N_JOBS = 30
swarm_dir = BASE_PATH / "swarm_dir/lmg"
cmd_path = swarm_dir / "cmds.swarm"
if cmd_path.exists():
    cmd_path.unlink()
cmd_path.parent.mkdir(exist_ok=True, parents=True)
for impact_type in ["parent", "self"]:
    config_dict = parent_config if impact_type == "parent" else self_config
    df_path = get_df_path(
        df_dir=BASE_PATH / "data",
        llama_model="Llama-3.3-70B-Instruct",
        agg_type=config_dict["agg_type"],
        desc_seg_type=config_dict["desc_seg_type"],
        impact_type=impact_type,
    )
    cmd = create_lmg_cmd(
        config_dict=config_dict,
        n_cv_repeats=10,
        n_cv_splits=3,
        seed=SEED,
        n_lmg_samples=5000,
        df_path=df_path,
        n_jobs=N_JOBS,
        export_path=BASE_PATH / f"lmg_results/{impact_type}.parquet",
    )
    cmd_list.append(cmd)
cmd_path.write_text("\n".join(cmd_list))

Sequentially locally

In [ ]:
# import subprocess
# for iteration, cmd in enumerate(cmd_list):
#     print(f"{iteration +1}/{len(cmd_list)}...")
#     print(cmd)
#     subprocess.run(cmd, shell=True, check=True)

Run swarm

In [ ]:
run_lmg_swarm(
    cmd_path=cmd_path,
    swarm_time="0:10:00",
    gb=8,
    num_threads=N_JOBS + 4,
    log_dir=swarm_dir / "logs",
    partition="quick",
    n_batch=1,
)

Load results

In [ ]:
import pandas as pd

res_dir = BASE_PATH / "lmg_results"
parent_lmg = pd.read_parquet(res_dir / "parent.parquet")
parent_lmg = parent_lmg.sort_values("outcome")
self_lmg = pd.read_parquet(res_dir / "self.parquet")
self_lmg = self_lmg.sort_values("outcome")

In [ ]:
from src.coef_visual import plot_heatmap

combined_summary = pd.merge(
    parent_lmg["mean"], self_lmg["mean"], on="outcome", suffixes=("_parent", "_self")
)
plot_heatmap(
    pivot=combined_summary,
    vmin=-0.05,
    vmax=0.05,
    title="Avg increase in % explained variance",
    fmt=".2%",
    xlabel="Cohort",
)